<a href="https://colab.research.google.com/github/noviansyah972/DataScience2026_240401010138_NoviansyahRamadhan/blob/main/Pertemuan3_NoviansyahRamadhan_240401010132.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Identitas Mahasiswa

**Nama Lengkap:** Noviansyah Ramadhan

**NIM:** 240401010132

**Kelas:** IF-403

In [1]:
import pandas as pd
import numpy as py
from pandas import json_normalize
from scipy.stats import mstats

In [5]:
df = pd.read_csv("https://drive.google.com/uc?id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo")
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [6]:
df.isnull().sum()

,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


In [8]:
#1 hapus duplikat
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

Setelah hapus duplikat: (130, 7)


In [13]:
#2 Normalisasi string
df['kota']    = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()
df.head()

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,Ygy,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,sedang


In [14]:
#3 Imputasi missing value
df['luas_m2']     = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta']  = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar']       = df['kamar'].fillna(df['kamar'].mode()[0])
df.head()


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,1.0,1995,bagus
2,3,249.7,895.0,Depok,1.0,1983,baik
3,4,49.7,178.0,Ygy,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,sedang


In [19]:
for col in ['harga_juta', 'luas_m2']:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    outlier = df[
        (df[col] < lower_fence) |
        (df[col] > upper_fence)
    ]

    print("="*50)
    print(f"Kolom : {col}")
    print(f"Q1           : {Q1}")
    print(f"Q3           : {Q3}")
    print(f"IQR          : {IQR}")
    print(f"Lower Fence  : {lower_fence}")
    print(f"Upper Fence  : {upper_fence}")
    print(f"Jumlah Outlier : {len(outlier)}")
    print(outlier[[col]])

Kolom : harga_juta
Q1           : 380.5
Q3           : 916.0
IQR          : 535.5
Lower Fence  : -422.75
Upper Fence  : 1719.25
Jumlah Outlier : 0
Empty DataFrame
Columns: [harga_juta]
Index: []
Kolom : luas_m2
Q1           : 101.6
Q3           : 266.15
IQR          : 164.54999999999998
Lower Fence  : -145.225
Upper Fence  : 512.9749999999999
Jumlah Outlier : 0
Empty DataFrame
Columns: [luas_m2]
Index: []


In [20]:
#5 Validasi dan ekspor
assert df.isnull().sum().sum()    == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print ('Shape akhir:', df.shape)
df.to_csv('housing_dirty.csv', index=False)
print('Dataset bersih tersimpan')



Shape akhir: (130, 7)
Dataset bersih tersimpan


In [21]:
#6 Akses API Json dan simpan respons
import requests
import pandas as pd
from pandas import json_normalize
URL      = "https://jsonplaceholder.typicode.com/users"

response = requests.get(URL, timeout=10)

if response.status_code == 200:
  data = response.json()
  df = json_normalize(data,sep='_')
  print("Data berhasil diambil")
  print(df.head())
else:
    print("Data gagal diambil")

Data berhasil diambil
   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address_street address_suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    010-692-6593 x09125  anastasia.net      Victor Plains     Suite 879   
2         1-463-123-4447    ramiro.info  Douglas Extension     Suite 847   
3      493-170-9623 x156       kale.biz        Hoeger Mall      Apt. 692   
4          (254)954-1289   demarco.info       Skiles Walks     Suite 351   

    address_city address_zipcode address_geo_lat address_geo_lng  \
0    Gwenborough      92998-3874        -37.3159         81.14